In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

In [2]:
IMG_H, IMG_W = 32, 128
CHANNELS     = 1
 
# Karakter yang relevan untuk data keuangan struk
# Fokus: angka, huruf, simbol umum di struk
CHARACTERS = list(
    "0123456789"
    "abcdefghijklmnopqrstuvwxyz"
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    " .,:-/()%"
)
NUM_CLASSES = len(CHARACTERS) + 1
 
# Kelas anotasi dataset
LABEL_CLASSES = [
    "harga_satuan",
    "nama_produk",
    "QTY",
    "tanggal",
    "total_harga_barang",
    "total_transaksi",
]
 
print("=" * 60)
print("TASK 1: EKSPERIMEN ARSITEKTUR MODEL OCR")
print("=" * 60)
print(f"Input per crop : ({IMG_H}, {IMG_W}, {CHANNELS})")
print(f"Jumlah kelas   : {NUM_CLASSES} karakter")
print(f"Label dataset  : {LABEL_CLASSES}")
print(f"TensorFlow     : {tf.__version__}")
print()

TASK 1: EKSPERIMEN ARSITEKTUR MODEL OCR
Input per crop : (32, 128, 1)
Jumlah kelas   : 72 karakter
Label dataset  : ['harga_satuan', 'nama_produk', 'QTY', 'tanggal', 'total_harga_barang', 'total_transaksi']
TensorFlow     : 2.21.0



In [3]:
def build_model_v1():
    """
    V1 - Ringan: 2 CNN + 1 BiLSTM
    Cocok untuk region sederhana: angka (harga, qty, total)
    """
    inputs = keras.Input(shape=(IMG_H, IMG_W, CHANNELS), name="input")
 
    x = layers.Conv2D(32, (3,3), padding="same", activation="relu")(inputs)
    x = layers.MaxPooling2D((2,2))(x)
 
    x = layers.Conv2D(64, (3,3), padding="same", activation="relu")(x)
    x = layers.MaxPooling2D((2,2))(x)
 
    new_h = IMG_H // 4
    new_w = IMG_W // 4
    x = layers.Reshape((new_w, new_h * 64))(x)
 
    x = layers.Bidirectional(
            layers.LSTM(64, return_sequences=True))(x)
    output = layers.Dense(
            NUM_CLASSES, activation="softmax", name="output")(x)
 
    return keras.Model(inputs, output, name="OCR_v1_Ringan")
 
 
def build_model_v2():
    """
    V2 - Sedang: 3 CNN + 2 BiLSTM (REKOMENDASI)
    Seimbang antara akurasi dan kecepatan.
    Cocok untuk semua kelas: nama_produk, tanggal, harga, total
    """
    inputs = keras.Input(shape=(IMG_H, IMG_W, CHANNELS), name="input")
 
    # CNN Block 1
    x = layers.Conv2D(32, (3,3), padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.25)(x)
 
    # CNN Block 2
    x = layers.Conv2D(64, (3,3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.25)(x)
 
    # CNN Block 3
    x = layers.Conv2D(128, (3,3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,1))(x)
    x = layers.Dropout(0.25)(x)
 
    new_h = IMG_H // 8
    new_w = IMG_W // 4
    x = layers.Reshape((new_w, new_h * 128))(x)
    x = layers.Dense(64, activation="relu")(x)
 
    x = layers.Bidirectional(
            layers.LSTM(128, return_sequences=True, dropout=0.25))(x)
    x = layers.Bidirectional(
            layers.LSTM(64,  return_sequences=True, dropout=0.25))(x)
 
    output = layers.Dense(
            NUM_CLASSES, activation="softmax", name="output")(x)
 
    return keras.Model(inputs, output, name="OCR_v2_Sedang")
 
def build_model_v3():
    """
    V3 - Dalam: 4 CNN + 2 BiLSTM
    Untuk teks kompleks seperti nama_produk panjang
    """
    inputs = keras.Input(shape=(IMG_H, IMG_W, CHANNELS), name="input")
 
    x = layers.Conv2D(32, (3,3), padding="same", activation="relu")(inputs)
    x = layers.Conv2D(32, (3,3), padding="same", activation="relu")(x)
    x = layers.MaxPooling2D((2,2))(x)
 
    x = layers.Conv2D(64, (3,3), padding="same", activation="relu")(x)
    x = layers.Conv2D(64, (3,3), padding="same", activation="relu")(x)
    x = layers.MaxPooling2D((2,2))(x)
 
    x = layers.Conv2D(128, (3,3), padding="same", activation="relu")(x)
    x = layers.MaxPooling2D((2,1))(x)
 
    new_h = IMG_H // 8
    new_w = IMG_W // 4
    x = layers.Reshape((new_w, new_h * 128))(x)
    x = layers.Dense(128, activation="relu")(x)
 
    x = layers.Bidirectional(
            layers.LSTM(128, return_sequences=True))(x)
    x = layers.Bidirectional(
            layers.LSTM(64,  return_sequences=True))(x)
 
    output = layers.Dense(
            NUM_CLASSES, activation="softmax", name="output")(x)
 
    return keras.Model(inputs, output, name="OCR_v3_Dalam")

In [4]:
models = {
    "v1 - Ringan (2CNN+1BiLSTM)": build_model_v1,
    "v2 - Sedang (3CNN+2BiLSTM)": build_model_v2,
    "v3 - Dalam  (4CNN+2BiLSTM)": build_model_v3,
}
 
print(f"{'Arsitektur':<32} {'Params':>12} {'Output Shape':>22}")
print("-" * 70)
 
dummy = np.zeros((1, IMG_H, IMG_W, CHANNELS), dtype=np.float32)
 
for nama, builder in models.items():
    m      = builder()
    params = m.count_params()
    out    = m.predict(dummy, verbose=0).shape
    print(f"{nama:<32} {params:>12,} {str(out):>22}")
 
print()
print("REKOMENDASI : v2 - Sedang")
print("  → Cukup powerful untuk baca semua kelas")
print("  → Tidak terlalu berat untuk backend MoneyLens")
print()
 
# Detail arsitektur v2
print("=" * 60)
print("DETAIL ARSITEKTUR V2 (Rekomendasi):")
print("=" * 60)
build_model_v2().summary()

Arsitektur                             Params           Output Shape
----------------------------------------------------------------------
v1 - Ringan (2CNN+1BiLSTM)            323,528            (1, 32, 72)
v2 - Sedang (3CNN+2BiLSTM)            497,672            (1, 32, 72)
v3 - Dalam  (4CNN+2BiLSTM)            641,320            (1, 32, 72)

REKOMENDASI : v2 - Sedang
  → Cukup powerful untuk baca semua kelas
  → Tidak terlalu berat untuk backend MoneyLens

DETAIL ARSITEKTUR V2 (Rekomendasi):


Model: "OCR_v2_Sedang"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 32, 128, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 32, 128, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 32, 128, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 16, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 16, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 16, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 16, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 8, 32, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 8, 32, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 8, 32, 128)     │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 8, 32, 128)     │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 4, 32, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 4, 32, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_3 (Reshape)             │ (None, 32, 512)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32, 64)         │        32,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_5 (Bidirectional) │ (None, 32, 256)        │       197,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_6 (Bidirectional) │ (None, 32, 128)        │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 32, 72)         │         9,288 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 497,672 (1.90 MB)

 Trainable params: 497,224 (1.90 MB)

 Non-trainable params: 448 (1.75 KB)

In [6]:
print()
print("=" * 60)
print("INFORMASI DEVICE")
print("=" * 60)
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"GPU tersedia : {[g.name for g in gpus]}")
    print(f"Menggunakan  : GPU")
    device = "GPU"
else:
    print(f"GPU          : Tidak tersedia")
    print(f"Menggunakan  : CPU")
    device = "CPU"


INFORMASI DEVICE
GPU          : Tidak tersedia
Menggunakan  : CPU


In [7]:
import json, os
 
BASE_DIR   = r"D:\program vscode\MoneyLens\ai\Dataset_ocr"
hasil = {
    "rekomendasi"  : "v2",
    "nama_model"   : "OCR_v2_Sedang",
    "arsitektur"   : "3CNN + 2BiLSTM + CTC",
    "device"       : device,
    "img_h"        : IMG_H,
    "img_w"        : IMG_W,
    "channels"     : CHANNELS,
    "num_classes"  : NUM_CLASSES,
    "params": {
        "v1": 323528,
        "v2": 497672,
        "v3": 641320,
    }
}
 
out_json = os.path.join(BASE_DIR, "task1_hasil.json")
with open(out_json, "w") as f:
    json.dump(hasil, f, indent=2)
 
print(f"\n💾 Hasil eksperimen disimpan: {out_json}")
print(f"   → Akan dibaca oleh Task 3 untuk evaluasi")


💾 Hasil eksperimen disimpan: D:\program vscode\MoneyLens\ai\Dataset_ocr\task1_hasil.json
   → Akan dibaca oleh Task 3 untuk evaluasi
